In [16]:

import json
import itertools
import os
from openai import OpenAI
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

# ============================================================
# CONFIGURAÇÕES
# ============================================================
CAMINHO_ARQUIVO = r'C:\Users\Thales Henrique\Documents\trabalho LLM\topic_generation\experiment_1_original_order\topics_final.json'

LIMIAR_SIMILARIDADE = 0.5

# Modelo da API (utilize o identificador disponibilizado à sua conta)
MODELO = "gpt-6.1-sol"

ARQUIVO_PROMPT = "../prompts/prompt_mesclagem.txt"
ARQUIVO_SAIDA = "taxonomia_mesclada.json"


# ============================================================
# 1. CARREGAR DADOS E GERAR EMBEDDINGS
# ============================================================
with open(CAMINHO_ARQUIVO, 'r', encoding='utf-8') as f:
    data = json.load(f)

topicos = data.get("topics", [])

textos_para_embedding = [
    f"{t['label']}: {t['description']}" for t in topicos
]

print("A carregar o modelo de embeddings...")
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("\nA gerar embeddings para os tópicos...")
embeddings = modelo.encode(
    textos_para_embedding,
    convert_to_tensor=True,
    show_progress_bar=True
)


# ============================================================
# 2. CALCULAR SIMILARIDADE
# ============================================================
similaridades = util.cos_sim(embeddings, embeddings)
pares_possiveis = list(itertools.combinations(range(len(topicos)), 2))
pares_similares = []

print("\nA verificar pares com alta similaridade...")

for i, j in tqdm(pares_possiveis, desc="A comparar pares", unit="par"):
    sim_score = similaridades[i][j].item()

    if sim_score >= LIMIAR_SIMILARIDADE:
        pares_similares.append({
            "score": sim_score,
            "topico_1": topicos[i],
            "topico_2": topicos[j]
        })

# Ordenar da maior para a menor similaridade
pares_similares = sorted(
    pares_similares,
    key=lambda x: x['score'],
    reverse=True
)

print(f"\nEncontrados {len(pares_similares)} pares suspeitos de redundância.")


# ============================================================
# 3. CONSTRUÇÃO DO PROMPT
# ============================================================

MERGE_SYSTEM_PROMPT = """
Você é um pesquisador especialista em análise temática.

Você receberá uma lista de tópicos e pares candidatos à mesclagem,
identificados previamente por similaridade semântica.


Sua tarefa é analisar todos os tópicos conjuntamente e produzir
uma consolidação global, seguindo estas regras:

1. Considere todos os pares em conjunto, pois um mesmo tópico
   pode aparecer em diferentes pares. Não tome decisões isoladas.

2. Mescle tópicos que apresentem proximidade temática relevante,
   mesmo que abordem aspectos diferentes de um mesmo assunto.
   Não exija equivalência de significado ou que os tópicos sejam
   praticamente sinônimos. Considere se eles podem ser
   representados por uma categoria temática mais abrangente,
   coerente e informativa.

3. Não presuma que a similaridade é transitiva: tópicos conectados
   por pares semelhantes não precisam pertencer ao mesmo grupo.
   Avalie cada possível grupo conjuntamente, verificando se todos
   os seus integrantes podem ser representados por um conceito
   abrangente e tematicamente coerente.

4. Cada tópico original deve pertencer a exatamente um grupo,
   que pode conter um ou mais tópicos. Não produza sobreposições.

5. Para grupos com mais de um tópico, crie um rótulo conciso
   e uma descrição abrangente que represente os aspectos
   compartilhados e complementares dos tópicos agrupados,
   sem introduzir assuntos que não estejam presentes nos originais.

6. Evite manter tópicos separados quando houver uma possibilidade
   razoável de agrupá-los em uma categoria temática coerente.
   Preserve individualmente apenas os tópicos que apresentem
   um foco suficientemente distinto dos demais.

7. Preserve os IDs originais e apresente uma justificativa breve
   para cada grupo formado ou tópico mantido separado.

8. Retorne exclusivamente um objeto JSON válido, seguindo
   rigorosamente o schema fornecido. Não inclua comentários,
   explicações ou blocos Markdown fora do JSON.
"""


# ============================================================
# 3.1. SCHEMA JSON DE SAÍDA
# ============================================================

SCHEMA_MESCLAGEM = {
    "type": "object",
    "properties": {
        "groups": {
            "type": "array",
            "description": "Lista de grupos resultantes da consolidação global.",
            "items": {
                "type": "object",
                "properties": {
                    "topic_ids": {
                        "type": "array",
                        "description": "IDs dos tópicos originais pertencentes ao grupo.",
                        "items": {
                            "anyOf": [
                                {"type": "integer"},
                                {"type": "string"}
                            ]
                        }
                    },
                    "label": {
                        "type": "string",
                        "description": "Rótulo conciso que representa o grupo."
                    },
                    "description": {
                        "type": "string",
                        "description": "Descrição abrangente do significado temático do grupo."
                    },
                    "justification": {
                        "type": "string",
                        "description": "Justificativa breve para a mesclagem ou manutenção separada."
                    }
                },
                "required": [
                    "topic_ids",
                    "label",
                    "description",
                    "justification"
                ],
                "additionalProperties": False
            }
        }
    },
    "required": ["groups"],
    "additionalProperties": False
}


# ============================================================
# 3.2. CONTEXTUALIZAÇÃO DO DOMÍNIO DE SAÚDE
# ============================================================

CONTEUDO_DOMINIO = (
    "DOMÍNIO E CONTEXTO DE SAÚDE:\n"
    "Os tópicos abaixo foram extraídos de títulos de vídeos do YouTube em português "
    "sobre medicamentos para emagrecimento (ex.: Ozempic, Mounjaro, "
    "etc.). Leve em consideração as especificidades clínicas, "
    "farmacológicas, comportamentais e sociais deste nicho ao decidir se dois tópicos "
    "possuem equivalência temática central ou apenas compartilham o assunto geral de saúde.\n"
)


# ============================================================
# 3.3. FORMATAR A LISTA COMPLETA DE TODOS OS TÓPICOS ORIGINAIS
# ============================================================

texto_todos_topicos = ""

for t in topicos:
    texto_todos_topicos += (
        f"[{t['topic_id']}] {t['label']}\n"
        f"    Descrição: {t['description']}\n"
    )


# ============================================================
# 3.4. FORMATAR AS SUGESTÕES DE PARES COM ALTA SIMILARIDADE
# ============================================================

texto_pares = ""

for par in pares_similares:
    t1 = par['topico_1']
    t2 = par['topico_2']

    texto_pares += (
        f"- [{t1['topic_id']}] e [{t2['topic_id']}] "
        f"(Similaridade: {par['score']:.4f})\n"
    )


# ============================================================
# 3.5. MONTAR O USER PROMPT UNIFICADO
# ============================================================

USER_PROMPT = f"""
{CONTEUDO_DOMINIO}
{texto_todos_topicos}

Abaixo estão os PARES CANDIDATOS com alta similaridade semântica para sua análise conjunta:
{texto_pares}

Por favor, forneça a consolidação global, garantindo que cada um dos IDs acima seja alocado em exatamente um grupo.
"""


# ============================================================
# 4. SALVAR O PROMPT FINAL
# ============================================================

with open(ARQUIVO_PROMPT, "w", encoding="utf-8") as f:
    #f.write("=== SYSTEM PROMPT ===\n")
    f.write(MERGE_SYSTEM_PROMPT.strip() + "\n\n")

    #f.write("=== USER PROMPT ===\n")
    f.write(USER_PROMPT.strip() + "\n\n")

    #f.write("=== JSON SCHEMA ===\n")
    f.write(json.dumps(SCHEMA_MESCLAGEM, ensure_ascii=False, indent=4))

print(f"\nPrompt final salvo em: {ARQUIVO_PROMPT}")


# ============================================================
# 5. CHAMADA À API DA OPENAI
# ============================================================

print("\nInicializando cliente da OpenAI...")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print(f"Enviando prompt para o modelo {MODELO}...")

response = client.chat.completions.create(
    model=MODELO,
    messages=[
        {
            "role": "system",
            "content": MERGE_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": USER_PROMPT
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "topic_merge",
            "strict": True,
            "schema": SCHEMA_MESCLAGEM
        }
    }
)

print("\nResposta recebida da API.")


# ============================================================
# 6. PROCESSAR E VALIDAR A RESPOSTA
# ============================================================

message = response.choices[0].message

if message.refusal:
    raise RuntimeError(f"O modelo recusou a solicitação: {message.refusal}")

if not message.content:
    raise RuntimeError("A API retornou uma resposta vazia.")

resultado = json.loads(message.content)

grupos = resultado["groups"]


# ============================================================
# 6.1. VALIDAR A ALOCAÇÃO DOS IDs ORIGINAIS
# ============================================================

ids_esperados = [str(t["topic_id"]) for t in topicos]

ids_recebidos = [
    str(topic_id)
    for grupo in grupos
    for topic_id in grupo["topic_ids"]
]

# IDs duplicados
ids_duplicados = [
    id_ for id_ in set(ids_recebidos)
    if ids_recebidos.count(id_) > 1
]

# IDs inexistentes
ids_inexistentes = [
    id_ for id_ in ids_recebidos
    if id_ not in ids_esperados
]

# IDs não alocados
ids_nao_alocados = [
    id_ for id_ in ids_esperados
    if id_ not in ids_recebidos
]

if ids_duplicados or ids_inexistentes or ids_nao_alocados:
    print("\nERRO NA VALIDAÇÃO DA TAXONOMIA!")

    print("IDs duplicados:", ids_duplicados)
    print("IDs inexistentes:", ids_inexistentes)
    print("IDs não alocados:", ids_nao_alocados)

    raise ValueError(
        "A resposta não alocou todos os tópicos originais exatamente uma vez."
    )

print("\nValidação dos IDs concluída com sucesso!")
print("Todos os tópicos originais foram alocados exatamente uma vez.")


# ============================================================
# 7. SALVAR A TAXONOMIA CONSOLIDADA
# ============================================================

with open(ARQUIVO_SAIDA, "w", encoding="utf-8") as f:
    json.dump(
        resultado,
        f,
        ensure_ascii=False,
        indent=4
    )

print(f"\nTaxonomia consolidada salva em: {ARQUIVO_SAIDA}")

print("\n" + "=" * 55)
print("RESUMO DA CONSOLIDAÇÃO")
print("=" * 55)

print(f"Tópicos originais: {len(topicos)}")
print(f"Pares candidatos: {len(pares_similares)}")
print(f"Grupos consolidados: {len(grupos)}")

grupos_mesclados = sum(
    1 for grupo in grupos if len(grupo["topic_ids"]) > 1
)

grupos_individuais = sum(
    1 for grupo in grupos if len(grupo["topic_ids"]) == 1
)

print(f"Grupos com mesclagem: {grupos_mesclados}")
print(f"Tópicos preservados individualmente: {grupos_individuais}")

print("\nProcesso concluído!")


A carregar o modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


A gerar embeddings para os tópicos...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


A verificar pares com alta similaridade...


A comparar pares: 100%|██████████| 210/210 [00:00<00:00, 70205.95par/s]


Encontrados 107 pares suspeitos de redundância.

Prompt final salvo em: prompt_mesclagem.txt

Inicializando cliente da OpenAI...
Enviando prompt para o modelo gpt-6.1-sol...



Resposta recebida da API.

Validação dos IDs concluída com sucesso!
Todos os tópicos originais foram alocados exatamente uma vez.

Taxonomia consolidada salva em: taxonomia_mesclada.json

RESUMO DA CONSOLIDAÇÃO
Tópicos originais: 21
Pares candidatos: 107
Grupos consolidados: 8
Grupos com mesclagem: 7
Tópicos preservados individualmente: 1

Processo concluído!
